# Phase II -- Interactive Detective Mystery (Claude API / Colab)

**No git clone, no GPU, no vLLM.** All files are embedded below as `%%writefile` cells; every LLM call goes through the Anthropic Claude API.

### What you need before running

1. An Anthropic API key -- get one at https://console.anthropic.com/
2. In Colab, click the **key icon** in the left sidebar ('Secrets'), add a
   new secret named **`ANTHROPIC_API_KEY`**, paste your key as the value,
   toggle 'Notebook access' on.
3. No GPU runtime required. CPU is fine because the model runs on
   Anthropic's servers.

### What this notebook does

1. Installs the `anthropic` SDK.
2. Writes every Python module + scripted transcript to `/content/`.
3. Pulls your API key from Colab secrets.
4. Generates one fixed mystery story + plan + world via Claude.
5. Assembles a novel-style `final_story.md`.
6. Replays both the successful and exception playthroughs.
7. Shows the drama-manager decision log.


## 1. Install the Anthropic SDK

(~10 seconds.)

In [ ]:
!pip install --quiet 'anthropic>=0.40'

## 2. Load your API key + pick the Claude model

Reads `ANTHROPIC_API_KEY` from Colab Secrets. If that fails, falls back to `getpass` so you can paste it once.

Default model: `claude-sonnet-4-5` -- fastest reasonable Sonnet. Use `claude-opus-4-7` if you want the strongest narrative quality (slower, pricier).

In [ ]:
import os, pathlib, getpass

# Create the working directories the scripts expect.
os.chdir('/content')
for d in ('data', 'logs', 'transcripts', 'scripts', 'tests'):
    pathlib.Path(d).mkdir(exist_ok=True)

# Pick a Claude model.
MODEL = 'claude-sonnet-4-5'   # or 'claude-opus-4-7' / 'claude-haiku-4-5-20251001'
os.environ['LLM_MODEL'] = MODEL

# Resolve the API key: prefer Colab Secrets, fall back to getpass.
if 'ANTHROPIC_API_KEY' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
        print('API key loaded from Colab Secrets.')
    except Exception as e:
        print('Secrets unavailable, asking interactively:', e)
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Paste ANTHROPIC_API_KEY: ')

assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY is still empty'
print('Model:', MODEL)
print('Workdir:', os.getcwd())


## 3. Write the Python modules

The next cells drop every source file into `/content/`. The only difference from the vLLM version is `llm_client.py` -- here it uses the Anthropic SDK with the exact same public surface (`chat`, `chat_simple`, `chat_json`), so none of the other modules need to change.

### `llm_client.py` (Claude backend)

In [ ]:
%%writefile llm_client.py
"""Drop-in replacement for llm_client.py that routes every call to the
Anthropic Claude API instead of a local vLLM server.

Same public surface (`chat`, `chat_simple`, `chat_json`, `health_check`,
`parse_json_safe`) as the vLLM-backed version, so none of the other modules
(phase1_story_generator, story_to_plan, world_builder, action_interpreter,
drama_manager, game_engine) need any change.

Env vars:
    ANTHROPIC_API_KEY   required
    LLM_MODEL           default claude-sonnet-4-5
"""
from __future__ import annotations

import json
import os
import re
import time
from typing import Any

import anthropic

_client: anthropic.Anthropic | None = None
_DEFAULT_MODEL = "claude-sonnet-4-5"


def _get_client() -> anthropic.Anthropic:
    global _client
    if _client is None:
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError(
                "ANTHROPIC_API_KEY is not set. In Colab, add it to "
                "'Secrets' (key icon) as ANTHROPIC_API_KEY and grant the "
                "notebook access."
            )
        _client = anthropic.Anthropic(api_key=api_key)
    return _client


def _resolve_model(model: str | None) -> str:
    return model or os.environ.get("LLM_MODEL", _DEFAULT_MODEL)


def _split_system(messages: list[dict[str, str]]) -> tuple[str | None, list[dict[str, str]]]:
    """OpenAI-style messages -> (system_prompt, claude_messages). Claude takes
    the system prompt as a separate top-level argument, not as a role=system
    message."""
    system_parts: list[str] = []
    claude_msgs: list[dict[str, str]] = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        if role == "system":
            if content:
                system_parts.append(content)
        else:
            claude_msgs.append({"role": role, "content": content})
    system = "\n\n".join(system_parts) if system_parts else None
    return system, claude_msgs


def chat(
    messages: list[dict[str, str]],
    *,
    model: str | None = None,
    max_tokens: int = 1024,
    temperature: float = 0.7,
    retries: int = 8,
    enable_thinking: bool = False,
    extra_body: dict[str, Any] | None = None,
    **kwargs: Any,
) -> str:
    """Send a chat request to Claude and return the assistant message content.

    Accepts the same keyword arguments as the vLLM-backed wrapper. vLLM-only
    options (`enable_thinking`, `extra_body`, `chat_template_kwargs`) are
    accepted and ignored so callers don't need to branch on backend.
    """
    _ = enable_thinking, extra_body, kwargs  # vLLM-only knobs; ignored

    resolved_model = _resolve_model(model)
    system, claude_msgs = _split_system(messages)
    last_err: Exception | None = None
    for attempt in range(retries):
        try:
            call_kwargs: dict[str, Any] = dict(
                model=resolved_model,
                max_tokens=max_tokens,
                temperature=temperature,
                messages=claude_msgs,
            )
            if system is not None:
                call_kwargs["system"] = system
            resp = _get_client().messages.create(**call_kwargs)
            text = "".join(
                block.text for block in resp.content if getattr(block, "type", None) == "text"
            )
            return text.strip()
        except (anthropic.APIConnectionError, anthropic.APITimeoutError,
                anthropic.InternalServerError, anthropic.RateLimitError) as err:
            last_err = err
            backoff = min(60.0, 2.0 ** attempt)
            time.sleep(backoff)
    raise RuntimeError(f"chat failed after {retries} retries: {last_err!r}")


def chat_simple(prompt: str, *, system: str | None = None, **kwargs: Any) -> str:
    """Convenience wrapper for a single-turn user prompt."""
    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return chat(messages, **kwargs)


_JSON_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)


def parse_json_safe(text: str) -> Any:
    """Strip markdown fences and parse JSON. Fallback: find first `{` / `[`."""
    text = text.strip()
    m = _JSON_FENCE.search(text)
    if m:
        text = m.group(1).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        for opener, closer in (("{", "}"), ("[", "]")):
            start = text.find(opener)
            end = text.rfind(closer)
            if start != -1 and end > start:
                try:
                    return json.loads(text[start : end + 1])
                except json.JSONDecodeError:
                    continue
        raise


def chat_json(
    prompt: str,
    *,
    system: str | None = None,
    max_tokens: int = 2048,
    temperature: float = 0.3,
    max_parse_retries: int = 3,
    **kwargs: Any,
) -> Any:
    """Ask for JSON, parse it, reprompt briefly if it fails to parse."""
    last_raw = ""
    for attempt in range(max_parse_retries):
        raw = chat_simple(
            prompt,
            system=system,
            max_tokens=max_tokens,
            temperature=temperature if attempt == 0 else max(0.1, temperature - 0.1 * attempt),
            **kwargs,
        )
        last_raw = raw
        try:
            return parse_json_safe(raw)
        except (json.JSONDecodeError, ValueError):
            continue
    raise ValueError(
        f"could not parse JSON after {max_parse_retries} attempts. Last raw:\n{last_raw[:400]}"
    )


def health_check() -> dict[str, Any]:
    """Return `{ok, backend, model}` -- fast smoke-test."""
    model = _resolve_model(None)
    try:
        out = chat_simple("Reply with just: ok", max_tokens=16, temperature=0.0, retries=2)
        return {"ok": True, "backend": "anthropic", "model": model, "reply": out}
    except Exception as err:  # noqa: BLE001
        return {"ok": False, "backend": "anthropic", "model": model, "error": repr(err)}


if __name__ == "__main__":
    print(json.dumps(health_check(), indent=2))


### `plan_types.py`

In [ ]:
%%writefile plan_types.py
"""Shared dataclasses for the plan representation.

An `Event` is a single step the detective (or an NPC) takes. It has explicit
preconditions and effects expressed as predicates over world-state subjects.
A `CausalLink` (producer, condition, consumer) indicates that the producer
event establishes `condition`, which the consumer event requires — so the
condition must remain true across the span between them. The drama manager
watches those spans.

World state is a flat dict[str, dict[str, Any]]: state[subject_id][attr].
Subjects are identifiers like "character.victoria_harrington",
"location.gallery_main_hall", "object.fountain_pen", "detective.inventory".
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any


@dataclass
class Condition:
    """A predicate over world state: `state[subject][attr] <op> value`.

    `op` ∈ {"==", "!=", "contains", "not_contains", ">=", "<="}.
    """
    subject: str
    attr: str
    op: str
    value: Any

    def evaluate(self, state: dict[str, dict[str, Any]]) -> bool:
        slot = state.get(self.subject, {}).get(self.attr)
        if self.op == "==":
            return slot == self.value
        if self.op == "!=":
            return slot != self.value
        if self.op == "contains":
            return slot is not None and self.value in slot
        if self.op == "not_contains":
            return slot is None or self.value not in slot
        if self.op == ">=":
            return slot is not None and slot >= self.value
        if self.op == "<=":
            return slot is not None and slot <= self.value
        raise ValueError(f"unknown op: {self.op}")

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Condition":
        return cls(subject=d["subject"], attr=d["attr"], op=d["op"], value=d["value"])


@dataclass
class Effect:
    """A world-state mutation.

    `op` ∈ {"set", "add", "remove"}. "add"/"remove" treat the slot as a
    set / list of items; "set" overwrites.
    """
    subject: str
    attr: str
    op: str
    value: Any

    def apply(self, state: dict[str, dict[str, Any]]) -> None:
        bucket = state.setdefault(self.subject, {})
        if self.op == "set":
            bucket[self.attr] = self.value
            return
        cur = bucket.get(self.attr)
        if self.op == "add":
            if isinstance(cur, list):
                if self.value not in cur:
                    cur.append(self.value)
            elif isinstance(cur, set):
                cur.add(self.value)
            else:
                bucket[self.attr] = [self.value]
        elif self.op == "remove":
            if isinstance(cur, list) and self.value in cur:
                cur.remove(self.value)
            elif isinstance(cur, set) and self.value in cur:
                cur.discard(self.value)
        else:
            raise ValueError(f"unknown op: {self.op}")

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Effect":
        return cls(subject=d["subject"], attr=d["attr"], op=d["op"], value=d["value"])


@dataclass
class Event:
    id: str
    actor: str
    verb: str
    args: list[str] = field(default_factory=list)
    location: str = ""
    preconditions: list[Condition] = field(default_factory=list)
    effects: list[Effect] = field(default_factory=list)
    reveals: list[str] = field(default_factory=list)
    description: str = ""
    narrative: str = ""
    source_plot_idx: int | None = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "actor": self.actor,
            "verb": self.verb,
            "args": self.args,
            "location": self.location,
            "preconditions": [c.to_dict() for c in self.preconditions],
            "effects": [e.to_dict() for e in self.effects],
            "reveals": self.reveals,
            "description": self.description,
            "narrative": self.narrative,
            "source_plot_idx": self.source_plot_idx,
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Event":
        return cls(
            id=d["id"],
            actor=d.get("actor", "detective"),
            verb=d.get("verb", "act"),
            args=list(d.get("args", [])),
            location=d.get("location", ""),
            preconditions=[Condition.from_dict(c) for c in d.get("preconditions", [])],
            effects=[Effect.from_dict(e) for e in d.get("effects", [])],
            reveals=list(d.get("reveals", [])),
            description=d.get("description", ""),
            narrative=d.get("narrative", ""),
            source_plot_idx=d.get("source_plot_idx"),
        )


@dataclass
class CausalLink:
    """(producer_event, condition, consumer_event)

    Condition must remain true from the moment `producer` executes until
    `consumer` executes. If something in between breaks it, we have an
    exception.
    """
    producer: str
    condition: Condition
    consumer: str

    def to_dict(self) -> dict[str, Any]:
        return {
            "producer": self.producer,
            "consumer": self.consumer,
            "condition": self.condition.to_dict(),
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "CausalLink":
        return cls(
            producer=d["producer"],
            consumer=d["consumer"],
            condition=Condition.from_dict(d["condition"]),
        )


@dataclass
class Plan:
    events: dict[str, Event] = field(default_factory=dict)
    order: list[tuple[str, str]] = field(default_factory=list)
    causal_links: list[CausalLink] = field(default_factory=list)
    initial_state: dict[str, dict[str, Any]] = field(default_factory=dict)
    goal: list[Condition] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "events": {eid: ev.to_dict() for eid, ev in self.events.items()},
            "order": [list(edge) for edge in self.order],
            "causal_links": [cl.to_dict() for cl in self.causal_links],
            "initial_state": self.initial_state,
            "goal": [c.to_dict() for c in self.goal],
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Plan":
        return cls(
            events={eid: Event.from_dict(e) for eid, e in d.get("events", {}).items()},
            order=[tuple(edge) for edge in d.get("order", [])],
            causal_links=[CausalLink.from_dict(cl) for cl in d.get("causal_links", [])],
            initial_state=d.get("initial_state", {}),
            goal=[Condition.from_dict(c) for c in d.get("goal", [])],
        )


### `phase1_story_generator.py`

In [ ]:
%%writefile phase1_story_generator.py
"""Phase I story generator — ported from Phase_I_Final_Story_Generator.ipynb.

All Claude API calls replaced with `llm_client.chat_simple` / `chat_json`.
Prompts preserved verbatim except where Claude-specific assumptions had to
be loosened for the Qwen chat template (see inline notes).

Usage:
    from phase1_story_generator import generate_full_story
    artifacts = generate_full_story("A poisoning murder at a 1920s London gallery")
    # artifacts["case_file"], ["complexities"], ["plot_points"], ["story_bible"], ["story_md"]
"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from llm_client import chat_json, chat_simple


# ---------------------------------------------------------------------------
# Checkpoint helpers
# ---------------------------------------------------------------------------
def save_checkpoint(data: Any, filename: str | Path) -> None:
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved: {path}")


def load_checkpoint(filename: str | Path) -> Any:
    return json.loads(Path(filename).read_text(encoding="utf-8"))


# ---------------------------------------------------------------------------
# Stage 1: case file
# ---------------------------------------------------------------------------
CASE_FILE_PROMPT = """You are a crime story architect. Generate a detailed murder mystery case file.
Output ONLY valid JSON, no markdown fences, no explanation.

{
  "criminal": {"name": str, "motive": str, "means": str, "opportunity": str},
  "victim": {"name": str, "background": str},
  "conspirators": [{"name": str, "role": str, "alibi": str}],
  "suspects": [{"name": str, "motive": str, "alibi": str}],
  "evidence": [
    {"id": str, "type": "physical|digital|testimonial",
     "description": str, "real_meaning": str, "steps_to_uncover": 2}
  ],
  "crime_timeline": [{"time": str, "event": str}],
  "solving_timeline": [
    {"step": int, "action": str, "target_evidence": [str], "max_actions": 3}
  ],
  "detective": {"name": str, "personal_stake": str, "deadline": str, "dire_consequence": str}
}

Generate at least 3 conspirators, 4 suspects, 8 evidence items, 6 solving steps."""


def generate_case_file(user_prompt: str) -> dict[str, Any]:
    full = f"{CASE_FILE_PROMPT}\n\nCrime context: {user_prompt}"
    return chat_json(full, max_tokens=3000, temperature=0.7)


# ---------------------------------------------------------------------------
# Stage 2: cover narrative / complexities
# ---------------------------------------------------------------------------
COMPLEXITIES_PROMPT = """Given real crime facts, generate a fabricated cover narrative.
Output ONLY valid JSON.

{
  "fake_suspect": {"name": str, "framing_reason": str},
  "planted_evidence": [{"description": str, "points_to": "fake_suspect"}],
  "false_testimonies": [{"witness": str, "claim": str}],
  "fake_timeline": [{"time": str, "event": str}],
  "evidence_fabrications": {"<evidence_id>": "fabricated explanation"},
  "conspirator_alibis": {"<name>": "alibi story"}
}

Rules:
- fake_suspect must NOT be the real criminal
- Every evidence id must appear in evidence_fabrications
- Alibis must be internally consistent"""


def generate_complexities(case_file: dict[str, Any]) -> dict[str, Any]:
    facts_summary = json.dumps(
        {
            "criminal": case_file["criminal"]["name"],
            "crime_timeline": case_file["crime_timeline"],
            "evidence_ids": [e["id"] for e in case_file["evidence"]],
            "conspirators": [c["name"] for c in case_file["conspirators"]],
        }
    )
    prompt = f"{COMPLEXITIES_PROMPT}\n\nReal facts: {facts_summary}"
    return chat_json(prompt, max_tokens=2000, temperature=0.7)


# ---------------------------------------------------------------------------
# Stage 3: meta-controller (plot points)
# ---------------------------------------------------------------------------
class StoryState:
    def __init__(self, case_file: dict[str, Any], complexities: dict[str, Any], max_points: int = 18) -> None:
        self.countdown = max_points + 3
        self.plot_points: list[dict[str, Any]] = []
        self.action_history: list[str] = []
        self.success_prob = 1.0
        self.evidence_progress = {e["id"]: 0 for e in case_file["evidence"]}
        self.alibi_status = {c["name"]: "unverified" for c in case_file["conspirators"]}
        self.closed_paths: set[str] = set()
        self.milestones_completed: set[int] = set()
        self.case_file = case_file
        self.complexities = complexities

    def tick(self) -> None:
        self.countdown -= 1
        self.success_prob = max(0.05, self.success_prob - 0.01)

    def is_done(self, min_points: int = 15) -> bool:
        all_milestones = len(self.milestones_completed) >= len(self.case_file["solving_timeline"])
        return all_milestones and len(self.plot_points) >= min_points


def _collision_detect(action: str, case_file: dict[str, Any]) -> dict[str, Any]:
    action_lower = action.lower()
    investigative_verbs = [
        "interview", "question", "investigate", "follow",
        "confront", "check", "ask", "visit", "examine", "search",
    ]
    for conspirator in case_file["conspirators"]:
        name_parts = conspirator["name"].lower().split()
        if any(part in action_lower for part in name_parts):
            if any(v in action_lower for v in investigative_verbs):
                return {"collision": True, "type": "conspirator", "target": conspirator["name"]}
    for evidence in case_file["evidence"]:
        keywords = set(evidence["description"].lower().split()) - {"the", "a", "an", "of", "in", "at", "to"}
        action_words = set(action_lower.split())
        if len(keywords & action_words) >= 2:
            return {"collision": True, "type": "evidence", "target": evidence["id"]}
    return {"collision": False, "type": None, "target": None}


def _check_extra_requirements(alibi_checks_done, multistep_clues_done, case_file):
    all_suspects = [s["name"] for s in case_file["suspects"]]
    alibis_covered = all(s in alibi_checks_done for s in all_suspects)
    multistep_done = sum(1 for v in multistep_clues_done.values() if v >= 2)
    return alibis_covered and multistep_done >= 3


def _decide_plot_type(state, alibi_checks_done, multistep_clues_done, case_file, iteration):
    suspects = [s["name"] for s in case_file["suspects"]]
    unchecked_suspects = [s for s in suspects if s not in alibi_checks_done]
    unfinished_clues = [eid for eid, steps in multistep_clues_done.items() if 0 < steps < 2]
    unstarted_clues = [eid for eid, steps in multistep_clues_done.items() if steps == 0]
    if unchecked_suspects and iteration % 3 == 1:
        return "alibi_check"
    if unfinished_clues:
        return "clue_followup"
    if unstarted_clues and iteration % 4 == 0:
        return "clue_start"
    if state.success_prob < 0.4:
        return "obstacle"
    return "progress"


def _generate_action(state, milestone, plot_type, case_file, complexities,
                     alibi_checks, multistep_clues, story_bible):
    suspects = case_file["suspects"]
    unchecked = [s for s in suspects if s["name"] not in alibi_checks]
    unfinished = [eid for eid, v in multistep_clues.items() if 0 < v < 2]
    unstarted = [eid for eid, v in multistep_clues.items() if v == 0]

    constraint = (
        f"CASE: Murder of {story_bible['victim_name']}.\n"
        f"Detective: {story_bible['detective_name']} and partner {story_bible['partner_name']}.\n"
        "All actions must relate to THIS murder case only. No hospitals, no unrelated crimes."
    )

    if plot_type == "alibi_check":
        target = unchecked[0] if unchecked else suspects[0]
        target_name = target["name"] if isinstance(target, dict) else target
        prompt = f"""{constraint}

Generate a detective action where the detective investigates the alibi of suspect: {target_name}
The action should involve contacting witnesses, checking records, or visiting locations related to the murder.
Recent actions (avoid repeating): {state.action_history[-3:]}
Output ONLY the action description (2-3 sentences)."""

    elif plot_type == "clue_followup":
        eid = unfinished[0]
        evidence = next((e for e in case_file["evidence"] if e["id"] == eid), None)
        prompt = f"""{constraint}

Generate a detective action that is a FOLLOW-UP step on this evidence from the murder scene:
Evidence: {evidence['description'] if evidence else eid}
This is step 2. The detective digs deeper (lab analysis, expert consultation, cross-referencing).
Output ONLY the action description (2-3 sentences)."""

    elif plot_type == "clue_start":
        eid = unstarted[0] if unstarted else list(multistep_clues.keys())[0]
        evidence = next((e for e in case_file["evidence"] if e["id"] == eid), None)
        prompt = f"""{constraint}

Generate a detective action discovering this murder evidence for the first time:
Evidence: {evidence['description'] if evidence else eid}
Step 1: detective notices something unusual but does NOT fully understand it yet.
Output ONLY the action description (2-3 sentences)."""
    else:
        prompt = f"""{constraint}

Generate a detective action for the murder investigation.
Current milestone: {milestone['action']}
Plot type: {plot_type}
Recent actions (avoid repeating): {state.action_history[-3:]}
Time pressure: {state.countdown} steps remaining
Output ONLY the action description (2-3 sentences)."""

    return chat_simple(prompt, max_tokens=150, temperature=0.8)


def _generate_narrative(action, collision, state, plot_type, case_file,
                        complexities, alibi_checks, multistep_clues, story_bible):
    fake_suspect = story_bible["fake_suspect"]
    constraint_header = f"""STRICT RULES - NEVER VIOLATE:
- This story is ONLY about the murder of: {story_bible['victim_name']}
- Detective: {story_bible['detective_name']} and partner {story_bible['partner_name']} (names never change)
- Real criminal (DO NOT reveal): {story_bible['real_criminal']}
- Fake suspect being framed: {fake_suspect} (names never change)
- Conspirators (exact names only): {story_bible['conspirator_names']}
- Key evidence: {story_bible['key_evidence']}
- Murder method: {story_bible['murder_method']}
- FORBIDDEN: Do NOT introduce hospitals, psychiatric wards, ambulances,
  unrelated victims, international crime networks, or any new murder cases.
- All red herrings must relate ONLY to the original murder of {story_bible['victim_name']}.
- Character names must NEVER change between chapters.

"""
    if plot_type == "alibi_check":
        instruction = (
            f"Write an alibi verification scene about the murder of {story_bible['victim_name']}. "
            "The detective checks a suspect's alibi for the night of the murder. "
            "A conspirator subtly provides false confirmation. "
            f"The detective ends up misled, suspicion points toward {fake_suspect}."
        )
    elif plot_type in ("clue_start", "clue_followup"):
        step = "initial discovery" if plot_type == "clue_start" else "deeper investigation"
        instruction = (
            f"Write a multi-step clue investigation ({step}) about the murder of {story_bible['victim_name']}. "
            "The detective examines physical evidence from the murder scene. "
            "Partial findings raise more questions. Do NOT reveal the full truth yet. "
            f"The clue should hint toward {fake_suspect} being guilty (misleadingly)."
        )
    elif collision["collision"]:
        instruction = (
            "Write a conspirator intervention scene. "
            f"A conspirator from the murder of {story_bible['victim_name']} smoothly misdirects the detective. "
            f"Detective almost gets close to real criminal {story_bible['real_criminal']}, then gets redirected toward {fake_suspect}."
        )
    elif plot_type == "obstacle":
        instruction = (
            f"Write an obstacle scene in the murder investigation of {story_bible['victim_name']}. "
            "A clue goes cold or a witness recants their statement about the murder. "
            "Detective frustrated, time running out to solve THIS case."
        )
    else:
        instruction = (
            f"Write a progress scene in the murder investigation of {story_bible['victim_name']}. "
            f"Small discovery points toward {fake_suspect} (the wrong person). "
            f"Dramatic irony: reader knows {story_bible['real_criminal']} is the real killer."
        )
    prompt = f"""{constraint_header}Write a suspenseful mystery plot point (2-3 paragraphs, 3rd person).
Detective's action: {action}
Writing instruction: {instruction}
Output ONLY the narrative paragraphs. Literary prose with dialogue."""
    return chat_simple(prompt, max_tokens=400, temperature=0.85)


def _update_tracking(plot_type, action, alibi_checks, multistep_clues, case_file):
    action_lower = action.lower()
    for suspect in case_file["suspects"]:
        name_parts = suspect["name"].lower().split()
        if any(p in action_lower for p in name_parts):
            if "alibi" in action_lower or plot_type == "alibi_check":
                if suspect["name"] not in alibi_checks:
                    alibi_checks[suspect["name"]] = "checked_false"
    for evidence in case_file["evidence"]:
        keywords = set(evidence["description"].lower().split()) - {"the", "a", "an", "of", "in"}
        if len(keywords & set(action_lower.split())) >= 2 or plot_type in ("clue_start", "clue_followup"):
            if plot_type == "clue_start" and multistep_clues.get(evidence["id"], 0) == 0:
                multistep_clues[evidence["id"]] = 1
                break
            if plot_type == "clue_followup" and multistep_clues.get(evidence["id"], 0) == 1:
                multistep_clues[evidence["id"]] = 2
                break


def _build_story_bible(case_file: dict[str, Any]) -> dict[str, Any]:
    return {
        "case_summary": f"Murder of {case_file['victim']['name']}",
        "victim_name": case_file["victim"]["name"],
        "detective_name": case_file["detective"]["name"],
        "partner_name": "Detective Martinez",
        "real_criminal": case_file["criminal"]["name"],
        "fake_suspect": case_file.get("fake_suspect", {}).get("name", "unknown"),
        "conspirator_names": [c["name"] for c in case_file["conspirators"]],
        "suspect_names": [s["name"] for s in case_file["suspects"]],
        "key_evidence": [e["description"] for e in case_file["evidence"][:3]],
        "murder_method": case_file["criminal"]["means"],
        "murder_location": case_file["victim"]["background"],
    }


def run_meta_controller(
    case_file: dict[str, Any],
    complexities: dict[str, Any],
    story_bible: dict[str, Any],
    min_points: int = 20,
    max_iter: int = 60,
) -> list[dict[str, Any]]:
    state = StoryState(case_file, complexities)
    milestone_idx = 0
    solving_tl = case_file["solving_timeline"]
    alibi_checks_done: dict[str, str] = {}
    multistep_clues_done = {e["id"]: 0 for e in case_file["evidence"]}

    for iteration in range(max_iter):
        if state.is_done(min_points) and _check_extra_requirements(
            alibi_checks_done, multistep_clues_done, case_file
        ):
            break
        state.tick()
        current_milestone = solving_tl[min(milestone_idx, len(solving_tl) - 1)]
        plot_type = _decide_plot_type(
            state, alibi_checks_done, multistep_clues_done, case_file, iteration
        )
        action = _generate_action(
            state, current_milestone, plot_type, case_file, complexities,
            alibi_checks_done, multistep_clues_done, story_bible,
        )
        state.action_history.append(action)
        collision = _collision_detect(action, case_file)
        narrative = _generate_narrative(
            action, collision, state, plot_type, case_file, complexities,
            alibi_checks_done, multistep_clues_done, story_bible,
        )
        state.plot_points.append(
            {
                "action": action, "narrative": narrative, "collision": collision,
                "prob": state.success_prob, "plot_type": plot_type,
            }
        )
        if collision["collision"]:
            state.success_prob = max(0.05, state.success_prob - 0.08)
        elif plot_type == "progress":
            state.success_prob = min(1.0, state.success_prob + 0.02)
        _update_tracking(plot_type, action, alibi_checks_done, multistep_clues_done, case_file)
        actions_on_ms = sum(
            1 for a in state.action_history
            if any(w in a.lower() for w in current_milestone["action"].lower().split()[:2])
        )
        if actions_on_ms >= current_milestone.get("max_actions", 3):
            state.milestones_completed.add(milestone_idx)
            milestone_idx = min(milestone_idx + 1, len(solving_tl) - 1)
        print(
            f"  Plot #{len(state.plot_points):2d} [{plot_type:15s}] "
            f"| prob={state.success_prob:.0%} "
            f"| collision={'YES' if collision['collision'] else 'no'}"
        )
    return state.plot_points


# ---------------------------------------------------------------------------
# Stage 4: assemble a complete novel-style markdown (Prologue + Chapters + Resolution + Epilogue)
# ---------------------------------------------------------------------------
_HANDCUFF_RE = re.compile(r"[^.]*handcuff[^.]*\.", re.IGNORECASE)


def _clean_plot_points(plot_points: list[dict[str, Any]], story_bible: dict[str, Any]) -> list[dict[str, Any]]:
    """Scrub premature arrests and leaked real-criminal names from individual
    plot-point narratives so the chapters can hide the truth until the end."""
    real = story_bible["real_criminal"]
    fake = story_bible["fake_suspect"]
    victim = story_bible["victim_name"]

    resolution_markers = (
        "you are under arrest", "i am arresting", "handcuffs clicked",
        "placed under arrest", "you're under arrest", "handcuffs snapped",
    )
    premature_rewrites = [
        (re.compile(rf"you['’]re under arrest(?: for)?[^.]*", re.IGNORECASE),
         f"you are a person of interest in the death of {victim}"),
        (re.compile(rf"you are under arrest(?: for)?[^.]*", re.IGNORECASE),
         f"you are a person of interest in the death of {victim}"),
        (re.compile(rf"i am arresting[^.]*", re.IGNORECASE),
         f"I need you to come in for further questioning regarding {victim}'s death"),
    ]

    cleaned: list[dict[str, Any]] = []
    for p in plot_points:
        narrative = p.get("narrative", "")
        lower = narrative.lower()

        for pat, repl in premature_rewrites:
            narrative = pat.sub(repl, narrative)
        if "handcuff" in lower:
            narrative = _HANDCUFF_RE.sub(
                "The detective made a mental note to arrange a formal interview.", narrative
            )
        # If the real criminal is named alongside any resolution marker, rename
        # them to the fake suspect — we must not reveal the killer in the body.
        if real and fake and real in narrative and any(mk in narrative.lower() for mk in resolution_markers):
            narrative = narrative.replace(real, fake)

        q = dict(p)
        q["narrative"] = narrative
        cleaned.append(q)
    return cleaned


def _stage_split(plot_points: list[dict[str, Any]]) -> list[tuple[int, str, str]]:
    """Return [(size, stage_title, stage_desc), ...] — the same five-stage
    breakdown used in the original notebook, scaled to the number of plot
    points produced this run."""
    total = max(len(plot_points), 5)
    s1 = max(2, int(total * 0.15))
    s2 = max(2, int(total * 0.20))
    s3 = max(4, int(total * 0.25))
    s4 = max(2, int(total * 0.20))
    s5 = max(1, total - s1 - s2 - s3 - s4)
    return [
        (s1, "Crime Scene Discovery",
         "Detective arrives at the scene, surveys the body, and notes the first physical clues."),
        (s2, "The Evidence Trail",
         "Multi-step investigation of key clues — examine, send to lab, cross-reference registries."),
        (s3, "Suspects and Misdirection",
         "Alibi checks for each suspect. Conspirators intervene and redirect suspicion toward the framed suspect."),
        (s4, "Closing In",
         "Detective narrows down suspects as evidence increasingly points (misleadingly) toward the framed suspect."),
        (s5, "Building the False Case",
         "Detective becomes convinced of the framed suspect's guilt; the final pieces are assembled."),
    ]


def assemble_story(
    case_file: dict[str, Any],
    plot_points: list[dict[str, Any]],
    story_bible: dict[str, Any],
    out_path: str | Path | None = None,
) -> str:
    """Generate a fluent novel-length markdown story from the plot points.

    Ported from the original Colab notebook (cell 8). The five-stage shape is
    preserved; prompts are rephrased so they work with Qwen's chat template
    and don't rely on Anthropic-specific formatting.
    """
    real = story_bible["real_criminal"]
    fake = story_bible["fake_suspect"]
    victim = story_bible["victim_name"]
    detective = story_bible["detective_name"]
    partner = story_bible.get("partner_name", "the detective's partner")
    conspirator_names = story_bible.get("conspirator_names", [])
    suspect_names = story_bible.get("suspect_names", [])
    murder_method = story_bible.get("murder_method", "unknown means")
    motive = case_file.get("criminal", {}).get("motive", "unknown motive")

    plot_points = _clean_plot_points(plot_points, story_bible)

    parts: list[str] = []

    prologue = chat_simple(
        f"""Write a mystery novel prologue revealing the TRUTH to the reader.

Constraints:
- Victim: {victim}
- Real killer: {real}
- Framed suspect (not the killer): {fake}
- Conspirators: {conspirator_names}
- Means: {murder_method}

Show what really happened on the night of the crime, atmospheric third-person prose,
220-260 words. Do not call it a prologue; open straight into the scene.""",
        max_tokens=500, temperature=0.75,
    )
    parts.append(f"# Prologue\n\n{prologue.strip()}")

    stage_defs = _stage_split(plot_points)
    idx = 0
    chapter_num = 1
    for size, stage_title, stage_desc in stage_defs:
        stage_points = plot_points[idx : idx + size]
        idx += size
        if not stage_points:
            continue
        # Two plot points per chapter.
        chunks = [stage_points[i : i + 2] for i in range(0, len(stage_points), 2)]
        for chunk in chunks:
            narratives = "\n\n".join(c["narrative"] for c in chunk)
            chapter = chat_simple(
                f"""Write Chapter {chapter_num} of a murder mystery novel.

STRICT RULES:
- The case is ONLY about the murder of: {victim}
- Detective: {detective} + partner {partner}
- Framed suspect being investigated: {fake}
- Real killer (do NOT reveal, do NOT arrest in this chapter): {real}
- Conspirator names (exact): {conspirator_names}
- Murder method: {murder_method}
- FORBIDDEN: hospitals, psychiatric wards, new crimes, new victims, unrelated plots.
- The detective must NOT solve the case or arrest anyone in this chapter.

Stage goal: {stage_desc}

Weave these two scenes together as one continuous chapter:

{narratives}

300-400 words. Literary prose with dialogue.""",
                max_tokens=650, temperature=0.8,
            )
            parts.append(f"# Chapter {chapter_num}: {stage_title}\n\n{chapter.strip()}")
            print(f"  Chapter {chapter_num} ({stage_title}) done")
            chapter_num += 1

    resolution = chat_simple(
        f"""Write a "Resolution" chapter for this murder mystery.

Setup:
- Victim: {victim}
- Detective: {detective}
- Detective WRONGLY arrests: {fake}
- Real killer is NOT caught: {real}
- Conspirator names: {conspirator_names}
- Suspect names: {suspect_names}
- Murder method: {murder_method}

Structure these four beats clearly:

1. EVIDENCE REVIEW — detective lays out the case against {fake} using concrete items.
2. ALIBI ELIMINATION — rule out each of these suspects by name and reason: {suspect_names}.
3. RECONSTRUCTION — "Here is what I believe happened that night..." walking step
   by step through a plausible (but wrong) sequence. Confident tone.
4. ARREST — detective confronts {fake}, who protests innocence. {real} watches
   from nearby, expression carefully composed.

450-520 words. Dramatic, confident tone.""",
        max_tokens=950, temperature=0.75,
    )
    parts.append(f"# The Resolution\n\n{resolution.strip()}")
    print("  Resolution chapter done")

    epilogue = chat_simple(
        f"""Write the epilogue for this murder mystery novel.

Constraints:
- Real killer: {real}
- Wrongly arrested: {fake}
- Victim: {victim}
- Conspirators: {conspirator_names}
- Motive: {motive}
- Means: {murder_method}

Four beats:
1. WRONG ARREST COMPLETE — {fake} is led away. Detective {detective} senses a
   small detail is off but dismisses it; the case is officially closed.
2. REAL KILLER'S HIDDEN CONFESSION — {real} mentally replays the truth, every
   gap filled: exact method, timing, disposal of evidence, role of each
   conspirator, and the motive ({motive}).
3. CONSPIRATORS' ROLES FULLY REVEALED (still in {real}'s thoughts).
4. THE LOOSE END — a minor character noticed something. They are too frightened
   to act tonight, but the truth has not been buried. End with a haunting
   one-line image.

320-360 words. Bittersweet, atmospheric tone.""",
        max_tokens=700, temperature=0.75,
    )
    parts.append(f"# Epilogue\n\n{epilogue.strip()}")
    print("  Epilogue done")

    story = "\n\n---\n\n".join(parts)
    if out_path:
        out = Path(out_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(story, encoding="utf-8")
        print(f"\nStory saved -> {out}  ({chapter_num - 1} chapters + Resolution + Epilogue)")
    return story


# ---------------------------------------------------------------------------
# Top-level driver
# ---------------------------------------------------------------------------
def generate_full_story(
    user_prompt: str = "A poisoning murder at a prestigious 1920s London art gallery opening",
    out_dir: str | Path = "data",
    min_points: int = 20,
) -> dict[str, Any]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Generating case file...")
    case_file = generate_case_file(user_prompt)
    save_checkpoint(case_file, out_dir / "case_file.json")

    print("Generating complexities...")
    complexities = generate_complexities(case_file)
    case_file["fake_suspect"] = complexities.get("fake_suspect", {})
    save_checkpoint(complexities, out_dir / "complexities.json")

    story_bible = _build_story_bible(case_file)
    save_checkpoint(story_bible, out_dir / "story_bible.json")

    print("Running meta-controller...")
    plot_points = run_meta_controller(case_file, complexities, story_bible, min_points=min_points)
    save_checkpoint(plot_points, out_dir / "plot_points.json")

    return {
        "case_file": case_file,
        "complexities": complexities,
        "story_bible": story_bible,
        "plot_points": plot_points,
    }


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd")

    gen = sub.add_parser("generate", help="Run Phase I from scratch")
    gen.add_argument("--prompt", default="A poisoning murder at a prestigious 1920s London art gallery opening")
    gen.add_argument("--out-dir", default="data")
    gen.add_argument("--min-points", type=int, default=20)

    asm = sub.add_parser("assemble", help="Assemble a markdown novel from an existing plot_points.json")
    asm.add_argument("--data-dir", default="data")
    asm.add_argument("--out", default="data/final_story.md")

    # Backward-compat: if no subcommand, default to generate with the given flags.
    parser.add_argument("--prompt", default=None)
    parser.add_argument("--out-dir", default=None)
    parser.add_argument("--min-points", type=int, default=None)

    args = parser.parse_args()
    if args.cmd == "assemble":
        data_dir = Path(args.data_dir)
        case_file = load_checkpoint(data_dir / "case_file.json")
        plot_points = load_checkpoint(data_dir / "plot_points.json")
        story_bible = load_checkpoint(data_dir / "story_bible.json")
        assemble_story(case_file, plot_points, story_bible, out_path=args.out)
    else:
        generate_full_story(
            args.prompt or "A poisoning murder at a prestigious 1920s London art gallery opening",
            args.out_dir or "data",
            args.min_points or 20,
        )


### `story_to_plan.py`

In [ ]:
%%writefile story_to_plan.py
"""Convert Phase I output into a partially ordered plan with causal links.

Phase I gives us:
  - case_file.json: evidence[], conspirators[], suspects[], solving_timeline[]
  - plot_points.json: 20-ish {action, narrative, plot_type, collision}

Phase I does NOT give us preconditions / effects / locations. We reconstruct
them here, one event at a time, with a structured LLM call and then sanity
filter the output to our typed dataclasses. A second pass builds causal
links by pattern-matching producer effects to consumer preconditions.

Output: plan.json (dict matching Plan.to_dict()).
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from llm_client import chat_json
from plan_types import CausalLink, Condition, Effect, Event, Plan


# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------
EVENT_EXTRACTION_SYSTEM = """You are a plan engineer for an interactive mystery game. Your job is to convert a detective's free-text action into a structured plan event with explicit preconditions and effects over a shared world state.

Use these subject-id conventions:
  character.<snake_case_name>   e.g. character.victoria_harrington
  location.<snake_case>         e.g. location.gallery_main_hall
  object.<snake_case>           e.g. object.fountain_pen
  evidence.<evidence_id>        e.g. evidence.E03
  detective                     (the player; has attrs: location, knowledge, inventory)

Always output valid JSON. Never add explanation outside the JSON."""


EVENT_EXTRACTION_PROMPT = """Convert this detective action into a structured event.

Context:
  victim: {victim}
  detective: {detective}
  suspects: {suspects}
  conspirators: {conspirators}
  available evidence ids: {evidence_ids}

Plot point index: {idx}
Plot type: {plot_type}
Action (free text): {action}
Narrative excerpt (for context only, do not copy verbatim): {narrative_short}

Output JSON with exactly these keys:
{{
  "verb": "interview | examine | visit | search | analyze | consult | confront | observe | reconstruct",
  "args": [string, ...],
  "location": "location.<snake_case>",
  "preconditions": [
    {{"subject": "...", "attr": "...", "op": "==", "value": ...}}, ...
  ],
  "effects": [
    {{"subject": "...", "attr": "...", "op": "set|add|remove", "value": ...}}, ...
  ],
  "reveals": ["evidence.<id>", ...]
}}

Rules:
- At least ONE precondition should reference detective.location == "<location>".
- At least ONE effect should update detective.knowledge (op=add) with what the detective now knows.
- If the action examines or finds a physical object, include an effect on that object.
- If the action interviews a character, include a precondition that the character is available
  (character.<name>.available == true) and an effect updating that character's
  alibi_status or willingness_to_talk.
- Use lowercase snake_case for all subject ids. Keep effects minimal and specific.
- Do NOT invent evidence ids that are not in the provided list."""


CAUSAL_LINK_PROMPT = """You are validating causal links between plan events.

Producer event (E_i) — effects: {producer_effects}
Consumer event (E_j) — preconditions: {consumer_preconditions}

Question: for each consumer precondition, does a producer effect directly establish it?
Output JSON: {{"links": [{{"condition": <precondition object>, "established_by_producer": true|false}}]}}

Return ONLY the JSON."""


# ---------------------------------------------------------------------------
# Extraction pipeline
# ---------------------------------------------------------------------------
def _slug(s: str) -> str:
    import re
    out = re.sub(r"[^a-zA-Z0-9]+", "_", s).strip("_").lower()
    return out or "unknown"


def _short(text: str, max_chars: int = 320) -> str:
    text = text.strip().replace("\n", " ")
    return text[:max_chars]


def extract_event_from_plot_point(
    idx: int,
    plot: dict[str, Any],
    case_file: dict[str, Any],
) -> Event:
    """Call the LLM once to infer structured event fields for one plot point."""
    ctx = {
        "victim": case_file["victim"]["name"],
        "detective": case_file["detective"]["name"],
        "suspects": [s["name"] for s in case_file["suspects"]],
        "conspirators": [c["name"] for c in case_file["conspirators"]],
        "evidence_ids": [e["id"] for e in case_file["evidence"]],
        "idx": idx,
        "plot_type": plot.get("plot_type", "progress"),
        "action": plot.get("action", ""),
        "narrative_short": _short(plot.get("narrative", "")),
    }
    prompt = EVENT_EXTRACTION_PROMPT.format(**ctx)
    try:
        parsed = chat_json(
            prompt,
            system=EVENT_EXTRACTION_SYSTEM,
            max_tokens=900,
            temperature=0.3,
        )
    except (ValueError, json.JSONDecodeError) as err:
        print(f"  [warn] event {idx} extraction failed ({err!r}); using fallback")
        parsed = _fallback_event_dict(plot)

    event_id = f"E{idx:02d}"
    preconditions = [Condition.from_dict(c) for c in parsed.get("preconditions", [])]
    effects = [Effect.from_dict(e) for e in parsed.get("effects", [])]
    # Guarantee the minimum contract: detective must be at the action location,
    # and the detective learns at least one thing from the event.
    location = parsed.get("location") or "location.unknown"
    if not any(pc.subject == "detective" and pc.attr == "location" for pc in preconditions):
        preconditions.append(Condition("detective", "location", "==", location))
    if not any(ef.subject == "detective" and ef.attr == "knowledge" for ef in effects):
        effects.append(Effect("detective", "knowledge", "add", f"learned_from_{event_id}"))

    return Event(
        id=event_id,
        actor="detective",
        verb=parsed.get("verb", "act"),
        args=list(parsed.get("args", [])),
        location=location,
        preconditions=preconditions,
        effects=effects,
        reveals=list(parsed.get("reveals", [])),
        description=plot.get("action", ""),
        narrative=plot.get("narrative", ""),
        source_plot_idx=idx,
    )


def _fallback_event_dict(plot: dict[str, Any]) -> dict[str, Any]:
    """Deterministic fallback when the LLM returns unparseable JSON."""
    return {
        "verb": "act",
        "args": [],
        "location": "location.unknown",
        "preconditions": [],
        "effects": [],
        "reveals": [],
    }


# ---------------------------------------------------------------------------
# Causal link derivation
# ---------------------------------------------------------------------------
def _conditions_match(effect: Effect, precondition: Condition) -> bool:
    """Does this effect establish this precondition?"""
    if effect.subject != precondition.subject or effect.attr != precondition.attr:
        return False
    if precondition.op == "==" and effect.op == "set":
        return effect.value == precondition.value
    if precondition.op == "contains" and effect.op == "add":
        return effect.value == precondition.value
    if precondition.op == "!=" and effect.op == "set":
        return effect.value != precondition.value
    return False


def derive_causal_links(events: list[Event]) -> list[CausalLink]:
    """Walk events in order; for each consumer precondition, find the nearest
    prior event whose effect establishes it. That pair becomes a causal link.

    We augment the strict effect↔precondition match with two heuristics so the
    plan isn't left link-less when the LLM's surface form drifts:

    1. **Reveal → reference**: if event E_i reveals evidence.X and a later
       event E_j references evidence.X (in args or reveals), add a link
       `(E_i, evidence.X.discovered == True, E_j)`. The evidence must stay
       discovered across the span — it being destroyed breaks the link.
    2. **Evidence analyzed → referenced**: if event E_i has an effect setting
       evidence.X.analyzed=true and a later event references X, add a link
       `(E_i, evidence.X.analyzed == True, E_j)`.
    """
    links: list[CausalLink] = []
    seen: set[tuple[str, str, str]] = set()  # (producer, consumer, attr) dedupe key

    def _add(producer: str, consumer: str, condition: Condition) -> None:
        key = (producer, consumer, f"{condition.subject}:{condition.attr}:{condition.value}")
        if key in seen:
            return
        seen.add(key)
        links.append(CausalLink(producer=producer, condition=condition, consumer=consumer))

    # Pass 1: strict effect↔precondition match.
    for j, consumer in enumerate(events):
        for pc in consumer.preconditions:
            for i in range(j - 1, -1, -1):
                producer = events[i]
                if any(_conditions_match(ef, pc) for ef in producer.effects):
                    _add(producer.id, consumer.id, pc)
                    break

    # Pass 2: reveal → reference heuristic.
    def _refs_to_evidence(ev: Event) -> set[str]:
        refs: set[str] = set(ev.reveals)
        for a in ev.args:
            if isinstance(a, str) and a.startswith("evidence."):
                refs.add(a)
        return refs

    for i, producer in enumerate(events):
        for ev_id in producer.reveals:
            if not ev_id.startswith("evidence."):
                continue
            # Consumer = next event that also references this evidence id.
            for j in range(i + 1, len(events)):
                if ev_id in _refs_to_evidence(events[j]):
                    _add(producer.id, events[j].id,
                         Condition(ev_id, "discovered", "==", True))

    # Pass 3: analyzed → reference.
    for i, producer in enumerate(events):
        for ef in producer.effects:
            if ef.subject.startswith("evidence.") and ef.attr == "analyzed" and ef.op == "set" and ef.value is True:
                for j in range(i + 1, len(events)):
                    if ef.subject in _refs_to_evidence(events[j]):
                        _add(producer.id, events[j].id,
                             Condition(ef.subject, "analyzed", "==", True))

    return links


# ---------------------------------------------------------------------------
# Initial state construction
# ---------------------------------------------------------------------------
def build_initial_state(case_file: dict[str, Any]) -> dict[str, dict[str, Any]]:
    """Seed world state with characters, evidence, and the detective.

    We don't yet know locations — world_builder.py will pin each character
    and evidence item to a starting location once the location graph exists.
    """
    state: dict[str, dict[str, Any]] = {}

    state["detective"] = {
        "location": "location.unknown",
        "knowledge": [],
        "inventory": [],
        "alive": True,
    }

    for s in case_file["suspects"]:
        sid = f"character.{_slug(s['name'])}"
        state[sid] = {
            "name": s["name"],
            "role": "suspect",
            "available": True,
            "alibi_status": "unverified",
            "alive": True,
            "willingness_to_talk": "neutral",
        }
    for c in case_file["conspirators"]:
        cid = f"character.{_slug(c['name'])}"
        state[cid] = {
            "name": c["name"],
            "role": "conspirator",
            "available": True,
            "alibi_status": "unverified",
            "alive": True,
            "willingness_to_talk": "evasive",
        }
    state[f"character.{_slug(case_file['victim']['name'])}"] = {
        "name": case_file["victim"]["name"],
        "role": "victim",
        "available": False,
        "alive": False,
    }

    for ev in case_file["evidence"]:
        eid = f"evidence.{ev['id']}"
        state[eid] = {
            "type": ev.get("type", "physical"),
            "description": ev.get("description", ""),
            "discovered": False,
            "analyzed": False,
            "destroyed": False,
            "location": "location.unknown",
        }

    return state


def build_goal(case_file: dict[str, Any]) -> list[Condition]:
    """Detective wins when they have identified the real criminal and linked
    the key evidence to that person (by name)."""
    real = _slug(case_file["criminal"]["name"])
    goal = [
        Condition("detective", "knowledge", "contains", f"identified:character.{real}"),
        Condition("detective", "knowledge", "contains", f"linked_evidence:character.{real}"),
    ]
    return goal


# ---------------------------------------------------------------------------
# Top-level driver
# ---------------------------------------------------------------------------
def build_plan(
    case_file: dict[str, Any],
    plot_points: list[dict[str, Any]],
    out_path: str | Path | None = None,
) -> Plan:
    print(f"Extracting events from {len(plot_points)} plot points...")
    events: list[Event] = []
    for idx, plot in enumerate(plot_points):
        ev = extract_event_from_plot_point(idx, plot, case_file)
        events.append(ev)
        print(f"  {ev.id} verb={ev.verb:10s} loc={ev.location:32s} "
              f"pre={len(ev.preconditions)} eff={len(ev.effects)} reveals={ev.reveals}")

    print("Deriving causal links...")
    links = derive_causal_links(events)
    print(f"  {len(links)} causal links derived")

    order = [(events[i].id, events[i + 1].id) for i in range(len(events) - 1)]

    plan = Plan(
        events={ev.id: ev for ev in events},
        order=order,
        causal_links=links,
        initial_state=build_initial_state(case_file),
        goal=build_goal(case_file),
    )

    if out_path:
        out = Path(out_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(plan.to_dict(), indent=2), encoding="utf-8")
        print(f"Saved: {out}")

    return plan


def load_plan(path: str | Path) -> Plan:
    return Plan.from_dict(json.loads(Path(path).read_text(encoding="utf-8")))


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--out", default="data/plan.json")
    args = parser.parse_args()

    data_dir = Path(args.data_dir)
    case_file = json.loads((data_dir / "case_file.json").read_text())
    plot_points = json.loads((data_dir / "plot_points.json").read_text())
    build_plan(case_file, plot_points, out_path=args.out)


### `world_builder.py`

In [ ]:
%%writefile world_builder.py
"""Build a graph-based world from a plan.

For every distinct location referenced in plan events, we create a node.
Adjacency is decided by a commonsense LLM call: two locations are adjacent
if someone would reasonably walk between them without crossing a third
place. If two *consecutive* events occur in locations that wouldn't be
adjacent in the real world (bedroom → restaurant), we insert one or two
intermediate locations so the user can actually traverse.

We also populate each location with the characters and evidence the plan
expects to be there, and initialize the detective at the story's first
location.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from llm_client import chat_json
from plan_types import Plan


@dataclass
class Location:
    id: str
    name: str
    description: str = ""
    adjacent: set[str] = field(default_factory=set)
    characters: set[str] = field(default_factory=set)
    evidence: set[str] = field(default_factory=set)

    def to_dict(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "name": self.name,
            "description": self.description,
            "adjacent": sorted(self.adjacent),
            "characters": sorted(self.characters),
            "evidence": sorted(self.evidence),
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Location":
        return cls(
            id=d["id"],
            name=d["name"],
            description=d.get("description", ""),
            adjacent=set(d.get("adjacent", [])),
            characters=set(d.get("characters", [])),
            evidence=set(d.get("evidence", [])),
        )


@dataclass
class World:
    locations: dict[str, Location] = field(default_factory=dict)
    starting_location: str = ""

    def to_dict(self) -> dict[str, Any]:
        return {
            "locations": {lid: loc.to_dict() for lid, loc in self.locations.items()},
            "starting_location": self.starting_location,
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "World":
        return cls(
            locations={lid: Location.from_dict(v) for lid, v in d["locations"].items()},
            starting_location=d.get("starting_location", ""),
        )


# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------
LOCATION_DESCRIBE_PROMPT = """Describe this mystery-story location in 1-2 sentences of atmospheric detail.
Location id: {loc_id}
Context (story era/setting): {era}
Output JSON: {{"name": "...", "description": "..."}}"""


ADJACENCY_PROMPT = """You are a commonsense spatial reasoner for a 1920s London murder mystery.

Below is a list of locations that appear in the story, in the order the
detective must visit them. For each pair of *consecutive* locations, answer
whether they would naturally be adjacent (someone could walk directly from
one to the other without traversing a third distinct place) and if not,
propose 1-2 intermediate locations that bridge them.

Locations: {locations}

Output JSON:
{{
  "pairs": [
    {{"a": "<id>", "b": "<id>",
      "adjacent": true|false,
      "intermediates": ["location.<snake_case>", ...]}},
    ...
  ],
  "extra_intermediate_descriptions": {{
    "location.<id>": "short description",
    ...
  }}
}}

Rules:
- Include one entry per consecutive pair.
- If adjacent is true, intermediates must be [].
- Only invent intermediate locations when the pair clearly wouldn't be adjacent."""


# ---------------------------------------------------------------------------
# Builder
# ---------------------------------------------------------------------------
def _describe_location(loc_id: str, era: str) -> dict[str, str]:
    try:
        return chat_json(
            LOCATION_DESCRIBE_PROMPT.format(loc_id=loc_id, era=era),
            max_tokens=200,
            temperature=0.6,
        )
    except Exception:
        pretty = loc_id.split(".", 1)[-1].replace("_", " ").title()
        return {"name": pretty, "description": f"A location in the case: {pretty}."}


def _analyze_adjacency(loc_ids: list[str]) -> dict[str, Any]:
    try:
        return chat_json(
            ADJACENCY_PROMPT.format(locations=loc_ids),
            max_tokens=1500,
            temperature=0.3,
        )
    except Exception as err:
        print(f"  [warn] adjacency prompt failed ({err!r}); falling back to linear chain")
        return {
            "pairs": [{"a": a, "b": b, "adjacent": True, "intermediates": []}
                      for a, b in zip(loc_ids[:-1], loc_ids[1:])],
            "extra_intermediate_descriptions": {},
        }


def build_world(plan: Plan, era: str = "1920s London") -> World:
    event_ids_in_order = sorted(plan.events.keys())
    loc_sequence: list[str] = []
    for eid in event_ids_in_order:
        loc = plan.events[eid].location or "location.unknown"
        loc_sequence.append(loc)

    # Dedupe while preserving order, but remember the sequential visits so
    # we can reason about consecutive-pair adjacency.
    unique_locs = list(dict.fromkeys(loc_sequence))
    print(f"Unique plan locations: {len(unique_locs)}")

    world = World()
    for lid in unique_locs:
        info = _describe_location(lid, era)
        world.locations[lid] = Location(id=lid, name=info.get("name", lid), description=info.get("description", ""))

    # Consecutive pairs along the event trajectory — these are the ones that
    # must be reachable by the player without teleporting.
    pairs: list[tuple[str, str]] = []
    for a, b in zip(loc_sequence[:-1], loc_sequence[1:]):
        if a != b and (a, b) not in pairs and (b, a) not in pairs:
            pairs.append((a, b))
    print(f"Consecutive location transitions to resolve: {len(pairs)}")

    ids_for_llm = list({lid for pair in pairs for lid in pair})
    adjacency = _analyze_adjacency(ids_for_llm) if ids_for_llm else {"pairs": [], "extra_intermediate_descriptions": {}}

    # Insert intermediates first so we can describe them before wiring edges.
    for mid_id, mid_desc in (adjacency.get("extra_intermediate_descriptions") or {}).items():
        if mid_id not in world.locations:
            pretty = mid_id.split(".", 1)[-1].replace("_", " ").title()
            world.locations[mid_id] = Location(id=mid_id, name=pretty, description=mid_desc)

    verdict_by_pair = {(p["a"], p["b"]): p for p in adjacency.get("pairs", []) if "a" in p and "b" in p}

    for a, b in pairs:
        verdict = verdict_by_pair.get((a, b)) or verdict_by_pair.get((b, a))
        if verdict and not verdict.get("adjacent", True):
            mids = [m for m in verdict.get("intermediates", []) if isinstance(m, str)]
            chain = [a, *mids, b]
            for mid in mids:
                if mid not in world.locations:
                    pretty = mid.split(".", 1)[-1].replace("_", " ").title()
                    world.locations[mid] = Location(id=mid, name=pretty, description=f"A passage between areas of the story.")
            for x, y in zip(chain[:-1], chain[1:]):
                world.locations[x].adjacent.add(y)
                world.locations[y].adjacent.add(x)
        else:
            world.locations[a].adjacent.add(b)
            world.locations[b].adjacent.add(a)

    # Seed contents: every event that reveals an evidence places that
    # evidence in the event's location; characters mentioned in args are
    # placed there too.
    for ev in plan.events.values():
        loc_id = ev.location
        if loc_id in world.locations:
            for rev in ev.reveals:
                world.locations[loc_id].evidence.add(rev)

    # Pin characters to the location of the first event that references them.
    for ev in plan.events.values():
        for arg in ev.args:
            if arg.startswith("character.") and ev.location in world.locations:
                # Only set if not already placed (first mention wins).
                already_placed = any(arg in loc.characters for loc in world.locations.values())
                if not already_placed:
                    world.locations[ev.location].characters.add(arg)

    world.starting_location = loc_sequence[0] if loc_sequence else next(iter(world.locations))
    if world.starting_location not in world.locations:
        # Unknown fallback — create a generic "case_briefing_room".
        world.locations["location.case_briefing_room"] = Location(
            id="location.case_briefing_room",
            name="Case Briefing Room",
            description="A quiet room where the detective sorts notes between outings.",
        )
        for lid in list(world.locations.keys()):
            if lid == "location.case_briefing_room":
                continue
            world.locations["location.case_briefing_room"].adjacent.add(lid)
            world.locations[lid].adjacent.add("location.case_briefing_room")
        world.starting_location = "location.case_briefing_room"

    # Also hook initial_state["detective"]["location"] to the start.
    plan.initial_state.setdefault("detective", {})["location"] = world.starting_location

    # Make sure every evidence entry has its "location" attr patched from the world.
    for loc in world.locations.values():
        for ev_id in loc.evidence:
            if ev_id in plan.initial_state:
                plan.initial_state[ev_id]["location"] = loc.id

    return world


def save_world(world: World, path: str | Path) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(world.to_dict(), indent=2), encoding="utf-8")
    print(f"Saved: {p}")


def load_world(path: str | Path) -> World:
    return World.from_dict(json.loads(Path(path).read_text(encoding="utf-8")))


if __name__ == "__main__":
    import argparse

    from story_to_plan import load_plan

    parser = argparse.ArgumentParser()
    parser.add_argument("--plan", default="data/plan.json")
    parser.add_argument("--out", default="data/world.json")
    args = parser.parse_args()

    plan = load_plan(args.plan)
    world = build_world(plan)
    save_world(world, args.out)
    # Persist the patched initial_state back into plan.json so game_engine
    # sees the resolved starting location.
    Path(args.plan).write_text(json.dumps(plan.to_dict(), indent=2), encoding="utf-8")
    print(f"Updated: {args.plan}")


### `action_interpreter.py`

In [ ]:
%%writefile action_interpreter.py
"""LLM-based parser that maps free-form user input into a structured action.

The parser returns a dict of the same shape as a plan `Event` (minus the id):
verb, args, location, preconditions, effects, reveals. The drama manager
then checks preconditions and either executes the action, flags it as an
exception, or triggers accommodation.

The parser *does not* know about causal links. Only about surface effects
it can infer from commonsense. The drama manager is the one that decides
whether those effects constitute a story exception.
"""
from __future__ import annotations

import json
from typing import Any

from llm_client import chat_json
from plan_types import Condition, Effect


PARSE_SYSTEM = """You are an action interpreter for a text adventure. Convert the player's short command into a structured action over the game world. Output only valid JSON. Never write explanation."""


PARSE_PROMPT = """World summary:
  Player location: {player_location}
  Adjacent locations: {adjacent}
  Objects here: {here_objects}
  Characters here: {here_characters}
  Known evidence ids: {evidence_ids}
  Player inventory: {inventory}
  Player knowledge snippets (partial): {knowledge_snippets}

Subject id conventions:
  detective                       the player
  character.<snake_name>
  location.<snake>
  evidence.<id>
  object.<snake>

Player's raw input (<=5 words expected, may exceed): "{raw}"

Produce this JSON:
{{
  "verb": "move | examine | interview | search | take | drop | use | analyze | confront | accuse | observe | wait | custom",
  "args": ["<subject_id or short phrase>", ...],
  "target_location": "<location id or empty string>",
  "preconditions": [{{"subject":"...", "attr":"...", "op":"==", "value":...}}, ...],
  "effects": [{{"subject":"...", "attr":"...", "op":"set|add|remove", "value":...}}, ...],
  "reveals": ["evidence.<id>", ...],
  "novel_state_vars": [
    {{"subject":"...", "attr":"...", "why":"short reason why this new state matters"}}, ...
  ],
  "plain_summary": "one-sentence description of what the player tries to do"
}}

Critical rules:
- For movement, set verb="move" and fill target_location with an adjacent id.
- For any physical manipulation that creates a NEW persistent state on an object
  (jamming, breaking, locking, hiding, destroying), list that state under
  "novel_state_vars" in addition to listing it in effects. This tells the
  drama manager the action introduced a state slot not previously modeled.
- If the player tries to "accuse" or "arrest" someone, add an effect updating
  detective.knowledge by adding "accused:<character.id>".
- Keep preconditions minimal — usually just where the player and target must be."""


def interpret_action(
    raw_input: str,
    world_summary: dict[str, Any],
) -> dict[str, Any]:
    """Parse one user command into a structured action dict.

    `world_summary` should include: player_location, adjacent (list of ids),
    here_objects, here_characters, evidence_ids, inventory, knowledge_snippets.
    """
    prompt = PARSE_PROMPT.format(
        raw=raw_input.strip(),
        player_location=world_summary.get("player_location", ""),
        adjacent=world_summary.get("adjacent", []),
        here_objects=world_summary.get("here_objects", []),
        here_characters=world_summary.get("here_characters", []),
        evidence_ids=world_summary.get("evidence_ids", []),
        inventory=world_summary.get("inventory", []),
        knowledge_snippets=world_summary.get("knowledge_snippets", [])[-6:],
    )
    try:
        parsed = chat_json(prompt, system=PARSE_SYSTEM, max_tokens=700, temperature=0.2)
    except (ValueError, json.JSONDecodeError):
        parsed = {
            "verb": "custom",
            "args": [raw_input.strip()],
            "target_location": "",
            "preconditions": [],
            "effects": [],
            "reveals": [],
            "novel_state_vars": [],
            "plain_summary": raw_input.strip(),
        }
    parsed.setdefault("novel_state_vars", [])
    parsed["_raw"] = raw_input
    return parsed


def structured_preconditions(parsed: dict[str, Any]) -> list[Condition]:
    out: list[Condition] = []
    for pc in parsed.get("preconditions", []):
        try:
            out.append(Condition.from_dict(pc))
        except Exception:  # noqa: BLE001 — malformed LLM output, skip this precondition only
            continue
    return out


def structured_effects(parsed: dict[str, Any]) -> list[Effect]:
    out: list[Effect] = []
    for ef in parsed.get("effects", []):
        try:
            out.append(Effect.from_dict(ef))
        except Exception:  # noqa: BLE001 — malformed LLM output, skip this effect only
            continue
    return out


### `drama_manager.py`

In [ ]:
%%writefile drama_manager.py
"""Intervention and Accommodation drama manager (Template 2).

Classifies every user action as constituent / consistent / exceptional and
repairs the plan when exceptions occur.

Key design points:

1. **Open-action problem**. The interpreter may produce effects on state
   variables that were never in the original plan (e.g. jam door with
   chair). Commonsense says those still threaten future events — a blocked
   door blocks anyone who must pass through it, even if no pre-declared
   causal link mentions "jammed". We solve this by running a *commonsense
   threat query* for every user action: we ask the LLM, given the effects
   and the remaining plan events, whether any future event would become
   impossible under normal physical/social reasoning. If yes, the action
   is exceptional even when no hard causal link is broken.

2. **Accommodation**. When something is exceptional, we remove the events
   that are no longer reachable and the links that depended on them, then
   ask the LLM to generate replacement events that re-establish the goal
   preconditions from the current world state. Replacement events are
   constrained to the existing world (characters + locations); they may
   modify *unrevealed* crime details but not the original crime outcome.

3. **Inspectable logs**. Every decision — classification, threat query,
   removed events, replacement events — is appended to an in-memory log
   and a JSONL file, so the final video can scroll the log for evidence of
   behind-the-scenes behavior.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from llm_client import chat_json
from plan_types import CausalLink, Condition, Effect, Event, Plan


# ---------------------------------------------------------------------------
# Log entry
# ---------------------------------------------------------------------------
@dataclass
class LogEntry:
    kind: str
    payload: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        return {"kind": self.kind, **self.payload}


# ---------------------------------------------------------------------------
# Commonsense threat / accommodation prompts
# ---------------------------------------------------------------------------
THREAT_SYSTEM = """You are a commonsense reasoner for an interactive detective story.
You assess whether a just-performed action makes future plan events impossible.
Output only valid JSON."""


THREAT_PROMPT = """User action (structured):
{parsed_action}

Remaining plan events (summarized; id, verb, args, location, key preconditions):
{remaining_events}

Active causal link conditions (must hold until the listed consumer event):
{active_links}

Question: does the user's action render any remaining event impossible or
meaningfully harder for a realistic actor (detective or NPC) to perform,
even if no hard causal link is formally broken?

Consider physical blocks ("jammed", "locked permanently", "broken"),
social blocks ("witness dead", "suspect fled the country"), and epistemic
blocks ("evidence destroyed").

Output JSON:
{{
  "threatened_events": [
    {{"event_id": "<id>",
      "reason": "short commonsense reason",
      "repairable": true|false}},
    ...
  ],
  "overall_classification_hint": "constituent | consistent | exceptional"
}}"""


ACCOMMODATION_SYSTEM = """You are a partial-order-plan repair agent. You output only valid JSON.
Your job: given a broken detective plan and the current world state, propose
replacement events that restore the detective's path to the goal. You may
modify unrevealed details of the crime but never its core outcome."""


ACCOMMODATION_PROMPT = """Current world snapshot (relevant slice): {world_snapshot}

Goal conditions: {goal}

Events just removed (unreachable): {removed_events}
Active causal links still standing: {surviving_links}
Characters present in the world: {characters}
Locations in the world: {locations}
Evidence still not destroyed: {live_evidence}

Produce replacement events that keep the story moving toward the goal.
Output JSON:
{{
  "replacement_events": [
    {{"id": "R01",
      "verb": "interview|examine|search|analyze|consult|observe|confront|reconstruct",
      "args": ["<subject id>", ...],
      "location": "location.<snake>",
      "preconditions": [{{"subject":"...", "attr":"...", "op":"==", "value":...}}, ...],
      "effects": [{{"subject":"...", "attr":"...", "op":"set|add|remove", "value":...}}, ...],
      "reveals": ["evidence.<id>", ...],
      "description": "one sentence the detective would say while doing this",
      "narrative": "one short paragraph of prose"
    }},
    ...
  ],
  "rationale": "2-3 sentences explaining how these events repair the plan"
}}

Constraints:
- Do NOT reveal the real criminal directly; preserve mystery.
- Prefer 1-3 replacement events (not a whole new plan).
- Each effect should mirror the contract of the removed events where possible,
  so the goal remains reachable."""


# ---------------------------------------------------------------------------
# Drama manager
# ---------------------------------------------------------------------------
class DramaManager:
    def __init__(self, plan: Plan, log_path: str | Path = "logs/drama.jsonl") -> None:
        self.plan = plan
        self.executed: list[str] = []
        self.remaining: list[str] = sorted(plan.events.keys())
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.log: list[LogEntry] = []
        self._next_repair_idx = 1

    # ---- logging ---------------------------------------------------------
    def _log(self, kind: str, **payload: Any) -> None:
        entry = LogEntry(kind=kind, payload=payload)
        self.log.append(entry)
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(entry.to_dict(), default=str) + "\n")

    # ---- active causal links ---------------------------------------------
    def active_links(self) -> list[CausalLink]:
        """Links whose producer has fired but whose consumer has not."""
        out: list[CausalLink] = []
        for cl in self.plan.causal_links:
            if cl.producer in self.executed and cl.consumer not in self.executed and cl.consumer in self.remaining:
                out.append(cl)
        return out

    # ---- classification --------------------------------------------------
    def classify(
        self,
        parsed_action: dict[str, Any],
        state: dict[str, dict[str, Any]],
    ) -> dict[str, Any]:
        """Return {classification, matched_event_id|None, threats, details}."""
        constituent_match = self._find_constituent_match(parsed_action)
        proposed_effects = [Effect.from_dict(e) for e in parsed_action.get("effects", []) if isinstance(e, dict)]
        hard_violations = self._hard_violations(proposed_effects)

        # Commonsense threat check — only needed if not already an obvious
        # hard violation, because the LLM call is expensive.
        soft_threats: list[dict[str, Any]] = []
        if not hard_violations and (proposed_effects or parsed_action.get("novel_state_vars")):
            soft_threats = self._commonsense_threats(parsed_action)

        classification = self._decide_classification(constituent_match, hard_violations, soft_threats)

        result = {
            "classification": classification,
            "matched_event_id": constituent_match,
            "hard_violations": [cl.to_dict() for cl in hard_violations],
            "soft_threats": soft_threats,
        }
        self._log(
            "classification",
            action=parsed_action,
            classification=classification,
            matched_event_id=constituent_match,
            hard_violations=[cl.to_dict() for cl in hard_violations],
            soft_threats=soft_threats,
        )
        return result

    def _find_constituent_match(self, parsed_action: dict[str, Any]) -> str | None:
        """Return the first remaining event whose verb + args substantially
        overlap with the parsed action."""
        raw = (parsed_action.get("_raw") or "").lower()
        parsed_verb = parsed_action.get("verb", "").lower()
        parsed_args_lower = {str(a).lower() for a in parsed_action.get("args", [])}
        for eid in self.remaining:
            ev = self.plan.events[eid]
            # Skip if the plan event requires a specific location the player
            # isn't at — that's clearly not this action.
            if parsed_verb and ev.verb.lower() != parsed_verb and not raw:
                continue
            ev_args_lower = {str(a).lower() for a in ev.args}
            if parsed_args_lower & ev_args_lower:
                return eid
            # Fallback: description substring match on the raw input.
            if raw and any(tok in ev.description.lower() for tok in raw.split() if len(tok) > 3):
                return eid
        return None

    def _hard_violations(self, proposed_effects: list[Effect]) -> list[CausalLink]:
        """Which active causal links are negated by these effects?"""
        violated: list[CausalLink] = []
        for cl in self.active_links():
            for ef in proposed_effects:
                if self._effect_negates_condition(ef, cl.condition):
                    violated.append(cl)
                    break
        return violated

    @staticmethod
    def _effect_negates_condition(effect: Effect, condition: Condition) -> bool:
        if effect.subject != condition.subject or effect.attr != condition.attr:
            return False
        if condition.op == "==" and effect.op == "set":
            return effect.value != condition.value
        if condition.op == "!=" and effect.op == "set":
            return effect.value == condition.value
        if condition.op == "contains" and effect.op == "remove":
            return effect.value == condition.value
        if condition.op == "not_contains" and effect.op == "add":
            return effect.value == condition.value
        return False

    def _commonsense_threats(self, parsed_action: dict[str, Any]) -> list[dict[str, Any]]:
        remaining_summary = [
            {
                "id": eid,
                "verb": self.plan.events[eid].verb,
                "args": self.plan.events[eid].args,
                "location": self.plan.events[eid].location,
                "preconditions": [pc.to_dict() for pc in self.plan.events[eid].preconditions],
            }
            for eid in self.remaining[:10]
        ]
        active = [cl.to_dict() for cl in self.active_links()]
        prompt = THREAT_PROMPT.format(
            parsed_action=json.dumps(parsed_action)[:1200],
            remaining_events=json.dumps(remaining_summary)[:2500],
            active_links=json.dumps(active)[:1500],
        )
        try:
            parsed = chat_json(prompt, system=THREAT_SYSTEM, max_tokens=600, temperature=0.2)
        except Exception as err:  # noqa: BLE001 — falling back is safer than crashing
            self._log("threat_query_failed", error=repr(err))
            return []
        return parsed.get("threatened_events", []) if isinstance(parsed, dict) else []

    @staticmethod
    def _decide_classification(
        constituent_match: str | None,
        hard_violations: list[CausalLink],
        soft_threats: list[dict[str, Any]],
    ) -> str:
        any_unrepairable_soft = any(not t.get("repairable", True) for t in soft_threats)
        if hard_violations or soft_threats:
            return "exceptional"
        if constituent_match:
            return "constituent"
        _ = any_unrepairable_soft  # reserved for future tiering
        return "consistent"

    # ---- execution -------------------------------------------------------
    def execute_constituent(self, event_id: str, state: dict[str, dict[str, Any]]) -> None:
        event = self.plan.events[event_id]
        for ef in event.effects:
            ef.apply(state)
        self.executed.append(event_id)
        if event_id in self.remaining:
            self.remaining.remove(event_id)
        self._log("executed_constituent", event_id=event_id)

    def apply_free_effects(
        self,
        parsed_action: dict[str, Any],
        state: dict[str, dict[str, Any]],
    ) -> None:
        for ef_dict in parsed_action.get("effects", []):
            try:
                Effect.from_dict(ef_dict).apply(state)
            except Exception:  # noqa: BLE001 — malformed LLM output, skip this effect only
                continue
        self._log("applied_free_effects", effects=parsed_action.get("effects", []))

    # ---- accommodation ---------------------------------------------------
    def accommodate(
        self,
        parsed_action: dict[str, Any],
        classification: dict[str, Any],
        state: dict[str, dict[str, Any]],
        world_locations: list[str],
        characters: list[str],
    ) -> dict[str, Any]:
        """Remove unreachable events + generate replacements."""
        threatened_ids = {t["event_id"] for t in classification.get("soft_threats", []) if "event_id" in t}
        for cl_dict in classification.get("hard_violations", []):
            threatened_ids.add(cl_dict.get("consumer"))
        threatened_ids.discard(None)

        removed_events: list[str] = []
        for eid in list(self.remaining):
            if eid in threatened_ids:
                self.remaining.remove(eid)
                removed_events.append(eid)
        self.plan.causal_links = [
            cl for cl in self.plan.causal_links
            if cl.producer in self.remaining or cl.producer in self.executed
            if cl.consumer in self.remaining
        ]

        # Ask the LLM for replacement events. Keep world snapshot lean.
        live_evidence = [k for k, v in state.items() if k.startswith("evidence.") and not v.get("destroyed", False)]
        prompt = ACCOMMODATION_PROMPT.format(
            world_snapshot=json.dumps(_compact_state(state))[:2500],
            goal=json.dumps([c.to_dict() for c in self.plan.goal]),
            removed_events=json.dumps([self.plan.events[eid].to_dict() for eid in removed_events if eid in self.plan.events])[:2000],
            surviving_links=json.dumps([cl.to_dict() for cl in self.plan.causal_links])[:1500],
            characters=json.dumps(characters),
            locations=json.dumps(world_locations),
            live_evidence=json.dumps(live_evidence),
        )
        try:
            parsed = chat_json(prompt, system=ACCOMMODATION_SYSTEM, max_tokens=1500, temperature=0.4)
        except Exception as err:  # noqa: BLE001 — worst case: skip repair and let goal remain reachable by other paths
            self._log("accommodation_failed", error=repr(err), removed_events=removed_events)
            return {"removed_events": removed_events, "replacement_events": [], "rationale": "repair failed"}

        replacement_dicts = parsed.get("replacement_events", []) if isinstance(parsed, dict) else []
        replacements: list[Event] = []
        for rd in replacement_dicts:
            rid = f"R{self._next_repair_idx:02d}"
            self._next_repair_idx += 1
            try:
                ev = Event(
                    id=rid,
                    actor="detective",
                    verb=rd.get("verb", "act"),
                    args=list(rd.get("args", [])),
                    location=rd.get("location", "location.unknown"),
                    preconditions=[Condition.from_dict(c) for c in rd.get("preconditions", [])],
                    effects=[Effect.from_dict(e) for e in rd.get("effects", [])],
                    reveals=list(rd.get("reveals", [])),
                    description=rd.get("description", ""),
                    narrative=rd.get("narrative", ""),
                )
            except Exception:  # noqa: BLE001 — skip malformed replacement, continue with others
                continue
            self.plan.events[rid] = ev
            self.remaining.append(rid)
            replacements.append(ev)

        self._log(
            "accommodation",
            removed_events=removed_events,
            replacement_event_ids=[e.id for e in replacements],
            rationale=parsed.get("rationale", "") if isinstance(parsed, dict) else "",
        )
        return {
            "removed_events": removed_events,
            "replacement_events": [e.to_dict() for e in replacements],
            "rationale": parsed.get("rationale", "") if isinstance(parsed, dict) else "",
        }

    # ---- goal check ------------------------------------------------------
    def goal_satisfied(self, state: dict[str, dict[str, Any]]) -> bool:
        return all(c.evaluate(state) for c in self.plan.goal)


def _compact_state(state: dict[str, dict[str, Any]], max_chars: int = 2000) -> dict[str, Any]:
    """Skip bulky fields so the LLM prompt doesn't blow past context."""
    compact: dict[str, Any] = {}
    for sid, slots in state.items():
        if sid.startswith("evidence.") or sid == "detective" or sid.startswith("character."):
            compact[sid] = slots
        else:
            compact[sid] = {k: v for k, v in slots.items() if k not in {"description"}}
    text = json.dumps(compact)
    if len(text) > max_chars:
        compact = {sid: slots for sid, slots in compact.items() if sid == "detective" or sid.startswith("evidence.")}
    return compact


### `game_engine.py`

In [ ]:
%%writefile game_engine.py
"""The text game engine: world state, I/O loop, action execution.

Responsibilities:
  - Own the mutable world state (initialized from plan.initial_state +
    world.locations).
  - Render a brief text description of the player's current location.
  - Call action_interpreter for every line of input.
  - Hand off to drama_manager for classification + accommodation.
  - Apply the resulting effects and narrate them back to the player.
  - Log everything.

The engine tries to stay dumb: it does not invent effects, it only applies
what the interpreter and drama manager produce. That keeps the decision
pipeline inspectable.
"""
from __future__ import annotations

import copy
import json
import re
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

from action_interpreter import interpret_action
from drama_manager import DramaManager
from llm_client import chat_simple
from plan_types import Effect, Plan
from world_builder import World


MAX_WORDS_PER_COMMAND = 8  # loose; spec suggests ~5


@dataclass
class EngineConfig:
    max_turns: int = 80
    prompt_label: str = "> "
    narrate_with_llm: bool = True
    log_dir: Path = Path("logs")


@dataclass
class TurnLog:
    turn: int
    raw: str
    parsed: dict[str, Any]
    classification: dict[str, Any]
    effects_applied: list[dict[str, Any]] = field(default_factory=list)
    accommodation: dict[str, Any] | None = None
    narration: str = ""


class GameEngine:
    def __init__(
        self,
        plan: Plan,
        world: World,
        config: EngineConfig | None = None,
    ) -> None:
        self.plan = plan
        self.world = world
        self.config = config or EngineConfig()
        self.config.log_dir.mkdir(parents=True, exist_ok=True)

        self.state: dict[str, dict[str, Any]] = copy.deepcopy(plan.initial_state)
        self.state.setdefault("detective", {})
        self.state["detective"]["location"] = world.starting_location

        self.drama = DramaManager(plan, log_path=self.config.log_dir / "drama.jsonl")
        self.turn_logs: list[TurnLog] = []
        self.turn_log_path = self.config.log_dir / "turns.jsonl"
        # Truncate the file at game start so replays are clean.
        self.turn_log_path.write_text("", encoding="utf-8")

    # ---------------- rendering ---------------------------------------
    def render_location(self) -> str:
        loc_id = self.state["detective"]["location"]
        loc = self.world.locations.get(loc_id)
        if loc is None:
            return f"[The detective stands in an unmapped place: {loc_id}.]"
        parts = [f"-- {loc.name} --", loc.description]
        if loc.adjacent:
            adj_names = ", ".join(self.world.locations[a].name for a in sorted(loc.adjacent) if a in self.world.locations)
            parts.append(f"Exits: {adj_names}")
        chars_here = [self._char_name(cid) for cid in loc.characters if self._is_available(cid)]
        if chars_here:
            parts.append("You see: " + ", ".join(chars_here))
        evidence_here = [
            self.state.get(eid, {}).get("description", eid)
            for eid in loc.evidence
            if not self.state.get(eid, {}).get("destroyed", False)
            and not self.state.get(eid, {}).get("analyzed", False)
        ]
        if evidence_here:
            parts.append("Nearby: " + ", ".join(evidence_here[:3]))
        return "\n".join(parts)

    def _char_name(self, cid: str) -> str:
        return self.state.get(cid, {}).get("name", cid)

    def _is_available(self, cid: str) -> bool:
        return self.state.get(cid, {}).get("alive", True) and self.state.get(cid, {}).get("available", True)

    def _world_summary_for_interpreter(self) -> dict[str, Any]:
        loc_id = self.state["detective"]["location"]
        loc = self.world.locations.get(loc_id)
        adj = sorted(loc.adjacent) if loc else []
        here_chars = [f"{cid} ({self._char_name(cid)})" for cid in (loc.characters if loc else [])]
        here_evidence = [
            eid for eid in (loc.evidence if loc else [])
            if not self.state.get(eid, {}).get("destroyed", False)
        ]
        return {
            "player_location": loc_id,
            "adjacent": adj,
            "here_objects": here_evidence,
            "here_characters": here_chars,
            "evidence_ids": [k for k in self.state if k.startswith("evidence.")],
            "inventory": self.state["detective"].get("inventory", []),
            "knowledge_snippets": self.state["detective"].get("knowledge", []),
        }

    # ---------------- input loop --------------------------------------
    def run(self, get_input=input, echo=print) -> str:
        echo(self.render_location())
        echo("")
        for turn in range(1, self.config.max_turns + 1):
            try:
                raw = get_input(self.config.prompt_label)
            except EOFError:
                echo("\n[input closed]")
                break
            if not raw.strip():
                continue
            if raw.strip().lower() in {"quit", "exit"}:
                echo("(detective signs off)")
                break
            raw = _truncate_input(raw)

            parsed = interpret_action(raw, self._world_summary_for_interpreter())
            classification = self.drama.classify(parsed, self.state)
            effects_applied: list[dict[str, Any]] = []
            accommodation_result: dict[str, Any] | None = None

            if classification["classification"] == "constituent":
                eid = classification["matched_event_id"]
                self.drama.execute_constituent(eid, self.state)
                effects_applied = [ef.to_dict() for ef in self.plan.events[eid].effects]
            elif classification["classification"] == "consistent":
                self.drama.apply_free_effects(parsed, self.state)
                self._apply_movement_if_any(parsed)
                effects_applied = parsed.get("effects", [])
            else:  # exceptional
                # Apply the free effects first so the world reflects what the
                # player actually did, then repair the plan around it.
                self.drama.apply_free_effects(parsed, self.state)
                self._apply_movement_if_any(parsed)
                effects_applied = parsed.get("effects", [])
                accommodation_result = self.drama.accommodate(
                    parsed,
                    classification,
                    self.state,
                    world_locations=list(self.world.locations.keys()),
                    characters=[s for s in self.state if s.startswith("character.")],
                )

            narration = self._narrate(parsed, classification, effects_applied, accommodation_result)

            self._log_turn(turn, raw, parsed, classification, effects_applied, accommodation_result, narration)
            echo(narration)
            echo("")

            if self.drama.goal_satisfied(self.state):
                echo(">>> The case is solved. <<<")
                return "solved"
        return "ended"

    # ---------------- helpers -----------------------------------------
    def _apply_movement_if_any(self, parsed: dict[str, Any]) -> None:
        target = parsed.get("target_location") or ""
        if not target:
            return
        cur_loc = self.state["detective"]["location"]
        cur = self.world.locations.get(cur_loc)
        if cur and target in cur.adjacent and target in self.world.locations:
            self.state["detective"]["location"] = target
            Effect("detective", "location", "set", target).apply(self.state)

    def _narrate(
        self,
        parsed: dict[str, Any],
        classification: dict[str, Any],
        effects_applied: list[dict[str, Any]],
        accommodation_result: dict[str, Any] | None,
    ) -> str:
        if not self.config.narrate_with_llm:
            return self._stub_narration(parsed, classification)
        tag = classification["classification"]
        event_hint = ""
        if tag == "constituent":
            eid = classification.get("matched_event_id")
            if eid and eid in self.plan.events:
                event_hint = self.plan.events[eid].narrative[:600]
        prompt = (
            "Write ONE short paragraph (3-5 sentences) of 1920s-noir prose narrating the outcome "
            "of the detective's action. Do NOT reveal the real killer. Stay grounded in what "
            "changed in the world.\n\n"
            f"Action summary: {parsed.get('plain_summary', parsed.get('_raw', ''))}\n"
            f"Classification: {tag}\n"
            f"Effects applied (structured): {json.dumps(effects_applied)[:600]}\n"
            f"Location: {self.state['detective']['location']}\n"
            f"Prior plot reference (only if constituent): {event_hint}\n"
            + ("Replacement events introduced: " + json.dumps(accommodation_result.get("replacement_events", []))[:600]
               if accommodation_result else "")
        )
        try:
            return chat_simple(prompt, max_tokens=220, temperature=0.7).strip()
        except Exception:
            return self._stub_narration(parsed, classification)

    @staticmethod
    def _stub_narration(parsed: dict[str, Any], classification: dict[str, Any]) -> str:
        return f"[{classification['classification']}] {parsed.get('plain_summary') or parsed.get('_raw', '')}"

    def _log_turn(
        self,
        turn: int,
        raw: str,
        parsed: dict[str, Any],
        classification: dict[str, Any],
        effects_applied: list[dict[str, Any]],
        accommodation_result: dict[str, Any] | None,
        narration: str,
    ) -> None:
        log = TurnLog(
            turn=turn, raw=raw, parsed=parsed, classification=classification,
            effects_applied=effects_applied, accommodation=accommodation_result, narration=narration,
        )
        self.turn_logs.append(log)
        with self.turn_log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(asdict(log), default=str) + "\n")


_WORD_RE = re.compile(r"\s+")


def _truncate_input(raw: str) -> str:
    tokens = _WORD_RE.split(raw.strip())
    if len(tokens) > MAX_WORDS_PER_COMMAND:
        tokens = tokens[:MAX_WORDS_PER_COMMAND]
    return " ".join(tokens)


### `main.py`

In [ ]:
%%writefile main.py
"""Entry point.

Modes:
  build    Run Phase I + story_to_plan + world_builder once; save everything.
  play     Load saved plan+world and launch the interactive game loop.
  replay   Read a scripted list of commands from a file and run them
           against the saved plan+world (for transcripts / regression tests).
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

# Reconfigure stdio to UTF-8 so LLM-generated unicode (em dashes, smart quotes,
# accented characters) never crashes the batch runner on locale-less SLURM shells.
for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding="utf-8", errors="replace")
    except AttributeError:
        pass

from game_engine import EngineConfig, GameEngine
from phase1_story_generator import assemble_story, generate_full_story, load_checkpoint
from story_to_plan import build_plan, load_plan
from world_builder import build_world, load_world, save_world


def cmd_build(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    if not args.skip_story or not (data_dir / "plot_points.json").exists():
        print("==> Phase I: generating story")
        generate_full_story(user_prompt=args.prompt, out_dir=data_dir, min_points=args.min_points)
    else:
        print("==> Reusing existing Phase I artifacts")

    case_file = load_checkpoint(data_dir / "case_file.json")
    plot_points = load_checkpoint(data_dir / "plot_points.json")

    print("==> Building plan")
    plan = build_plan(case_file, plot_points, out_path=data_dir / "plan.json")

    print("==> Building world")
    world = build_world(plan)
    save_world(world, data_dir / "world.json")
    # Persist plan again so detective.location is set correctly.
    (data_dir / "plan.json").write_text(json.dumps(plan.to_dict(), indent=2), encoding="utf-8")
    print(f"All artifacts saved to {data_dir}/")
    return 0


def cmd_play(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    plan = load_plan(data_dir / "plan.json")
    world = load_world(data_dir / "world.json")
    engine = GameEngine(plan, world, EngineConfig(log_dir=Path(args.log_dir)))
    status = engine.run()
    print(f"[game ended: {status}]")
    return 0


def cmd_assemble(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    case_file = load_checkpoint(data_dir / "case_file.json")
    plot_points = load_checkpoint(data_dir / "plot_points.json")
    story_bible = load_checkpoint(data_dir / "story_bible.json")
    assemble_story(case_file, plot_points, story_bible, out_path=args.out)
    return 0


def cmd_replay(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    plan = load_plan(data_dir / "plan.json")
    world = load_world(data_dir / "world.json")
    cmds = Path(args.script).read_text(encoding="utf-8").splitlines()
    cmds_iter = iter(cmds)

    def scripted_input(prompt: str) -> str:
        try:
            line = next(cmds_iter)
        except StopIteration:
            raise EOFError
        print(prompt + line)
        return line

    engine = GameEngine(plan, world, EngineConfig(log_dir=Path(args.log_dir)))
    status = engine.run(get_input=scripted_input)
    print(f"[replay ended: {status}]")
    return 0


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser()
    subs = parser.add_subparsers(dest="cmd", required=True)

    sp_build = subs.add_parser("build", help="Generate story + plan + world")
    sp_build.add_argument("--prompt", default="A poisoning murder at a prestigious 1920s London art gallery opening")
    sp_build.add_argument("--data-dir", default="data")
    sp_build.add_argument("--min-points", type=int, default=20)
    sp_build.add_argument("--skip-story", action="store_true",
                          help="Skip Phase I if data/plot_points.json already exists.")
    sp_build.set_defaults(func=cmd_build)

    sp_asm = subs.add_parser("assemble", help="Assemble a novel-style markdown from an existing plot_points.json")
    sp_asm.add_argument("--data-dir", default="data")
    sp_asm.add_argument("--out", default="data/final_story.md")
    sp_asm.set_defaults(func=cmd_assemble)

    sp_play = subs.add_parser("play", help="Launch interactive game")
    sp_play.add_argument("--data-dir", default="data")
    sp_play.add_argument("--log-dir", default="logs")
    sp_play.set_defaults(func=cmd_play)

    sp_replay = subs.add_parser("replay", help="Run a scripted transcript")
    sp_replay.add_argument("script")
    sp_replay.add_argument("--data-dir", default="data")
    sp_replay.add_argument("--log-dir", default="logs")
    sp_replay.set_defaults(func=cmd_replay)

    args = parser.parse_args(argv)
    return args.func(args)


if __name__ == "__main__":
    sys.exit(main())


### scripted transcripts

In [ ]:
%%writefile transcripts/success_run.txt
examine the body
look around
question the coat-check
examine the champagne flute
go to gallery office
search desk drawer
examine fountain pen
take fountain pen
go to lab
analyze the pen
go to gallery
interview victoria harrington
question dr fleming
check guest registry
cross reference records
question lord pemberton
interview agnes the attendant
examine witness video
reconstruct the timeline
confront victoria harrington
accuse victoria harrington


In [ ]:
%%writefile transcripts/exception_run.txt
examine the body
look around
examine champagne flute
smash the flute
go to gallery office
take fountain pen
drop pen in fireplace
go to coat check
interview agnes
threaten agnes
question victoria harrington
wedge morgue door shut
examine witness video
reconstruct the timeline
question dr fleming
accuse victoria harrington


## 4. Smoke test

One round-trip to confirm the API key works.

In [ ]:
import llm_client, importlib
importlib.reload(llm_client)   # in case the %%writefile cell was re-run
import json
print(json.dumps(llm_client.health_check(), indent=2))


## 5. Build the story + plan + world

Runs Phase I -> story_to_plan -> world_builder against Claude. Expect ~3-8 min with Sonnet (faster than a local 8B model). The artifacts land in `data/`.

In [ ]:
!python main.py build --data-dir data --min-points 15

## 6. Assemble a novel-style story

Produces `data/final_story.md`. ~2-4 min.

In [ ]:
!python main.py assemble --data-dir data --out data/final_story.md

Render inline:

In [ ]:
from IPython.display import Markdown, display
display(Markdown(pathlib.Path('data/final_story.md').read_text(encoding='utf-8')))


## 7. Replay a successful run

In [ ]:
!python main.py replay transcripts/success_run.txt --data-dir data --log-dir logs

## 8. Replay an exception run (drama manager accommodates)

In [ ]:
!python main.py replay transcripts/exception_run.txt --data-dir data --log-dir logs

## 9. Inspect the drama manager log

Every classification, commonsense threat check, removed event and replacement event is a JSON line in `logs/drama.jsonl`.

In [ ]:
import json
for line in pathlib.Path('logs/drama.jsonl').read_text().splitlines()[-20:]:
    entry = json.loads(line)
    kind = entry.pop('kind')
    print(kind, '-', json.dumps(entry)[:200])


## 10. (Optional) Interactive mode

Input format:
- <= 8 words per command
- Free-form natural language; Claude parses it into a structured action
- Exits are shown each turn; use those location names for movement
- Type `quit` or `exit` to end

Examples: `examine the body`, `go to gallery`, `interview eleanor voss`, `smash the flute`, `wedge morgue door shut`.

In [ ]:
from game_engine import GameEngine, EngineConfig
from story_to_plan import load_plan
from world_builder import load_world

plan = load_plan('data/plan.json')
world = load_world('data/world.json')
engine = GameEngine(plan, world, EngineConfig(log_dir=pathlib.Path('logs/interactive')))

def ask(prompt):
    return input(prompt)

status = engine.run(get_input=ask, echo=print)
print('\n=== game ended:', status, '===')
